# Diabetic Hospital Readmission Prediction
## Notebook 09 — Optuna Hyperparameter Optimization

### What This Notebook Does:
- Installs and imports Optuna Bayesian optimization library
- Loads the engineered dataset from Notebook 03
- Applies same 60/30/10 split and SMOTE 30% as Notebook 04
- Runs 100 Optuna trials for XGBoost optimization
- Runs 100 Optuna trials for LightGBM optimization
- Trains final optimized models with best parameters
- Compares Optuna results with manual tuning results
- Saves optimized models for deployment

### Why Optuna Instead of GridSearchCV?
GridSearchCV tests combinations in a fixed grid.
We previously tested only 12 combinations which is very limited.

Optuna uses Bayesian optimization — it learns which parameter
regions look promising and focuses the search there.
100 Optuna trials is more effective than 12 GridSearch trials
and takes roughly the same time.

### Output Files:
- models/xgb_optuna.pkl
- models/lgb_optuna.pkl
- models/optuna_best_params.pkl
- models/optuna_thresholds.pkl
- dashboard/optuna_comparison.csv

In [1]:
import subprocess
subprocess.run(['pip', 'install', 'optuna', '--quiet'])

import optuna
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import warnings
from sklearn.model_selection import (train_test_split,
                                     StratifiedKFold,
                                     cross_val_score)
from sklearn.metrics import (roc_auc_score, recall_score,
                             precision_score, f1_score,
                             roc_curve)
from xgboost import XGBClassifier
import lightgbm as lgb
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_SEED = 123

print(f'All libraries loaded!')
print(f'Optuna version: {optuna.__version__}')
print(f'Random seed:    {RANDOM_SEED}')

All libraries loaded!
Optuna version: 4.8.0
Random seed:    123


In [2]:
# Load engineered dataset
df = pd.read_csv('../data/processed/diabetic_engineered.csv')

# Load feature names and specialty means
model_features = joblib.load('../models/feature_names.pkl')
specialty_means = joblib.load('../models/specialty_means.pkl')

# Add specialty_risk from merged dataset
df_with_specialty = pd.read_csv('../data/processed/diabetic_with_sdoh.csv')
overall_mean = df['readmitted_30'].mean()
df['specialty_risk'] = df_with_specialty['medical_specialty'].map(
    specialty_means).fillna(overall_mean)

# Use model features
available = [f for f in model_features if f in df.columns]
X = df[available]
y = df['readmitted_30']

# Same 60/30/10 split as Notebook 04
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40,
    random_state=RANDOM_SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.25,
    random_state=RANDOM_SEED, stratify=y_temp
)

# Apply SMOTE 30% to training set only
smote = SMOTE(
    random_state=RANDOM_SEED,
    sampling_strategy=0.30
)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'Data loaded!')
print(f'  Features: {len(available)}')
print(f'  Training: {len(X_train_sm):,} rows (after SMOTE)')
print(f'  Validation: {len(X_val):,} rows')
print(f'  Test: {len(X_test):,} rows — SEALED')
print(f'\nClass balance after SMOTE:')
print(f'  Class 0: {(y_train_sm==0).sum():,} ({(y_train_sm==0).mean()*100:.1f}%)')
print(f'  Class 1: {(y_train_sm==1).sum():,} ({(y_train_sm==1).mean()*100:.1f}%)')

Data loaded!
  Features: 68
  Training: 49,682 rows (after SMOTE)
  Validation: 20,992 rows
  Test: 6,998 rows — SEALED

Class balance after SMOTE:
  Class 0: 38,217 (76.9%)
  Class 1: 11,465 (23.1%)


In [3]:
# 3-Fold CV for speed during Optuna search
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)

# XGBoost Objective Function
def xgb_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'scale_pos_weight': trial.suggest_int('scale_pos_weight', 5, 15),
        'eval_metric': 'logloss',
        'verbosity': 0,
        'random_state': RANDOM_SEED
    }
    model = XGBClassifier(**params)
    scores = cross_val_score(
        model, X_train_sm, y_train_sm,
        cv=cv, scoring='roc_auc', n_jobs=-1
    )
    return scores.mean()

# LightGBM Objective Function
def lgb_objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 500),
        'max_depth':         trial.suggest_int('max_depth', 3, 10),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 150),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'scale_pos_weight':  trial.suggest_int('scale_pos_weight', 5, 15),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.0, 1.0),
        'random_state': RANDOM_SEED,
        'verbose': -1
    }
    model = lgb.LGBMClassifier(**params)
    scores = cross_val_score(
        model, X_train_sm, y_train_sm,
        cv=cv, scoring='roc_auc', n_jobs=-1
    )
    return scores.mean()

print('Objective functions defined!')
print(f'\nXGBoost parameters being tuned (7):')
print(f'  n_estimators:     100 to 500')
print(f'  max_depth:        3 to 10')
print(f'  learning_rate:    0.01 to 0.3')
print(f'  subsample:        0.6 to 1.0')
print(f'  colsample_bytree: 0.6 to 1.0')
print(f'  min_child_weight: 1 to 10')
print(f'  scale_pos_weight: 5 to 15')
print(f'\nLightGBM parameters being tuned (10):')
print(f'  n_estimators:      100 to 500')
print(f'  max_depth:         3 to 10')
print(f'  learning_rate:     0.01 to 0.3')
print(f'  num_leaves:        20 to 150')
print(f'  subsample:         0.6 to 1.0')
print(f'  colsample_bytree:  0.6 to 1.0')
print(f'  min_child_samples: 5 to 50')
print(f'  scale_pos_weight:  5 to 15')
print(f'  reg_alpha:         0.0 to 1.0')
print(f'  reg_lambda:        0.0 to 1.0')

Objective functions defined!

XGBoost parameters being tuned (7):
  n_estimators:     100 to 500
  max_depth:        3 to 10
  learning_rate:    0.01 to 0.3
  subsample:        0.6 to 1.0
  colsample_bytree: 0.6 to 1.0
  min_child_weight: 1 to 10
  scale_pos_weight: 5 to 15

LightGBM parameters being tuned (10):
  n_estimators:      100 to 500
  max_depth:         3 to 10
  learning_rate:     0.01 to 0.3
  num_leaves:        20 to 150
  subsample:         0.6 to 1.0
  colsample_bytree:  0.6 to 1.0
  min_child_samples: 5 to 50
  scale_pos_weight:  5 to 15
  reg_alpha:         0.0 to 1.0
  reg_lambda:        0.0 to 1.0


Each objective function defines the search space for hyperparameters. trial.suggest_int finds the best integer value in a range. trial.suggest_float finds the best float value — log=True for learning_rate means Optuna searches on a logarithmic scale which is more appropriate since learning rates span several orders of magnitude. We use 3-fold CV instead of 5-fold to speed up the 100 trials — each trial trains 3 models so 100 trials = 300 model fits total.

In [5]:
# Run Optuna XGBoost Optimization
print('Running Optuna XGBoost optimization (100 trials)')

xgb_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED)
)
xgb_study.optimize(xgb_objective, n_trials=100)

print(f'\nXGBoost Optuna complete!')
print(f'  Best CV ROC-AUC: {xgb_study.best_value:.4f}')
print(f'\nBest parameters found:')
for k, v in xgb_study.best_params.items():
    print(f'  {k}: {v}')

Running Optuna XGBoost optimization (100 trials)

XGBoost Optuna complete!
  Best CV ROC-AUC: 0.8913

Best parameters found:
  n_estimators: 499
  max_depth: 10
  learning_rate: 0.08292019680569757
  subsample: 0.6065850351483025
  colsample_bytree: 0.985981607194762
  min_child_weight: 1
  scale_pos_weight: 7


We create an Optuna study with direction='maximize' because we want to maximize ROC-AUC. TPESampler is the Tree-structured Parzen Estimator algorithm — this is Optuna's default and most powerful sampler. It builds a probabilistic model of the objective function and samples new parameters from regions that are likely to improve the score. After 100 trials Optuna reports the best parameter combination found.

In [6]:
# Run Optuna LightGBM Optimization
print('Running Optuna LightGBM optimization (100 trials)')

lgb_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED)
)
lgb_study.optimize(lgb_objective, n_trials=100)

print(f'\nLightGBM Optuna complete!')
print(f'  Best CV ROC-AUC: {lgb_study.best_value:.4f}')
print(f'\nBest parameters found:')
for k, v in lgb_study.best_params.items():
    print(f'  {k}: {v}')

Running Optuna LightGBM optimization (100 trials)

LightGBM Optuna complete!
  Best CV ROC-AUC: 0.8839

Best parameters found:
  n_estimators: 438
  max_depth: 10
  learning_rate: 0.04620201864166716
  num_leaves: 83
  subsample: 0.8880426363593802
  colsample_bytree: 0.7050800343541103
  min_child_samples: 45
  scale_pos_weight: 5
  reg_alpha: 0.08162398418764394
  reg_lambda: 0.511674345354574


We run the same Optuna optimization for LightGBM. LightGBM has 10 parameters to tune including reg_alpha and reg_lambda regularization terms which XGBoost does not have in our search space. More parameters means a larger search space — Optuna handles this efficiently by focusing on the most promising regions. The TPESampler learns from XGBoost trials are separate from LightGBM trials — each study is independent.

### Train Final Optimized Models

In [7]:
# Train Final Models with Optuna Best Parameters
print('Training final models with Optuna best parameters')

# Best XGBoost
best_xgb = XGBClassifier(
    **xgb_study.best_params,
    eval_metric='logloss',
    verbosity=0
)
best_xgb.fit(X_train_sm, y_train_sm)

y_prob_xgb_opt = best_xgb.predict_proba(X_val)[:, 1]
fpr, tpr, thr  = roc_curve(y_val, y_prob_xgb_opt)
thr_xgb_opt    = thr[np.argmax(tpr - fpr)]
y_pred_xgb_opt = (y_prob_xgb_opt >= thr_xgb_opt).astype(int)

auc_xgb_opt  = roc_auc_score(y_val, y_prob_xgb_opt)
rec_xgb_opt  = recall_score(y_val, y_pred_xgb_opt)
prec_xgb_opt = precision_score(y_val, y_pred_xgb_opt)
f1_xgb_opt   = f1_score(y_val, y_pred_xgb_opt)

print(f'XGBoost Optuna Results (threshold={thr_xgb_opt:.4f}):')
print(f'  ROC-AUC:   {auc_xgb_opt:.4f}')
print(f'  Recall:    {rec_xgb_opt:.4f}')
print(f'  Precision: {prec_xgb_opt:.4f}')
print(f'  F1:        {f1_xgb_opt:.4f}')

# Best LightGBM
best_lgb = lgb.LGBMClassifier(
    **lgb_study.best_params,
    verbose=-1
)
best_lgb.fit(X_train_sm, y_train_sm)

y_prob_lgb_opt = best_lgb.predict_proba(X_val)[:, 1]
fpr, tpr, thr  = roc_curve(y_val, y_prob_lgb_opt)
thr_lgb_opt    = thr[np.argmax(tpr - fpr)]
y_pred_lgb_opt = (y_prob_lgb_opt >= thr_lgb_opt).astype(int)

auc_lgb_opt  = roc_auc_score(y_val, y_prob_lgb_opt)
rec_lgb_opt  = recall_score(y_val, y_pred_lgb_opt)
prec_lgb_opt = precision_score(y_val, y_pred_lgb_opt)
f1_lgb_opt   = f1_score(y_val, y_pred_lgb_opt)

print(f'\nLightGBM Optuna Results (threshold={thr_lgb_opt:.4f}):')
print(f'  ROC-AUC:   {auc_lgb_opt:.4f}')
print(f'  Recall:    {rec_lgb_opt:.4f}')
print(f'  Precision: {prec_lgb_opt:.4f}')
print(f'  F1:        {f1_lgb_opt:.4f}')

Training final models with Optuna best parameters
XGBoost Optuna Results (threshold=0.0335):
  ROC-AUC:   0.5893
  Recall:    0.6081
  Precision: 0.1110
  F1:        0.1878

LightGBM Optuna Results (threshold=0.2457):
  ROC-AUC:   0.6102
  Recall:    0.5534
  Precision: 0.1216
  F1:        0.1994


We train the final models using the best parameters found by Optuna. We pass the best_params dictionary directly using ** unpacking — this automatically sets all parameters without manually listing each one. We use the same optimal threshold approach as Notebook 04 — finding the threshold that maximizes Youden's J statistic from the ROC curve. This ensures a fair comparison with the manually tuned models.

### Comparison Table

In [8]:
# Optuna vs Manual Tuning Comparison
optuna_results = pd.DataFrame({
    'Model': [
        'XGBoost manual',
        'LightGBM manual',
        'XGBoost Optuna',
        'LightGBM Optuna'
    ],
    'ROC_AUC': [
        0.5953, 0.6068,
        auc_xgb_opt, auc_lgb_opt
    ],
    'Recall': [
        0.4408, 0.4594,
        rec_xgb_opt, rec_lgb_opt
    ],
    'Precision': [
        0.1260, 0.1275,
        prec_xgb_opt, prec_lgb_opt
    ],
    'F1': [
        0.1960, 0.1996,
        f1_xgb_opt, f1_lgb_opt
    ],
    'CV_AUC': [
        '0.8823', '0.8818',
        f'{xgb_study.best_value:.4f}',
        f'{lgb_study.best_value:.4f}'
    ],
    'Tuning': [
        'Manual', 'Manual',
        'Optuna 100 trials',
        'Optuna 100 trials'
    ]
}).round(4)

print('-' * 80)
print('OPTUNA vs MANUAL TUNING COMPARISON')
print('-' * 80)
print(optuna_results.to_string(index=False))
print('-' * 80)

best_auc = optuna_results.loc[optuna_results['ROC_AUC'].idxmax()]
best_rec = optuna_results.loc[optuna_results['Recall'].idxmax()]

print(f'\nBest ROC-AUC: {best_auc["Model"]} — {best_auc["ROC_AUC"]:.4f}')
print(f'Best Recall:  {best_rec["Model"]} — {best_rec["Recall"]:.4f}')

optuna_results.to_csv('../dashboard/optuna_comparison.csv', index=False)
print('\nOptuna comparison saved!')

--------------------------------------------------------------------------------
OPTUNA vs MANUAL TUNING COMPARISON
--------------------------------------------------------------------------------
          Model  ROC_AUC  Recall  Precision     F1 CV_AUC            Tuning
 XGBoost manual   0.5953  0.4408     0.1260 0.1960 0.8823            Manual
LightGBM manual   0.6068  0.4594     0.1275 0.1996 0.8818            Manual
 XGBoost Optuna   0.5893  0.6081     0.1110 0.1878 0.8913 Optuna 100 trials
LightGBM Optuna   0.6102  0.5534     0.1216 0.1994 0.8839 Optuna 100 trials
--------------------------------------------------------------------------------

Best ROC-AUC: LightGBM Optuna — 0.6102
Best Recall:  XGBoost Optuna — 0.6081

Optuna comparison saved!


In [9]:
# Save Optuna optimized models
joblib.dump(best_xgb, '../models/xgb_optuna.pkl')
joblib.dump(best_lgb, '../models/lgb_optuna.pkl')

# Save best parameters
best_params = {
    'xgb': xgb_study.best_params,
    'lgb': lgb_study.best_params
}
joblib.dump(best_params, '../models/optuna_best_params.pkl')

# Save optimal thresholds
optuna_thresholds = {
    'xgb_optuna': float(thr_xgb_opt),
    'lgb_optuna': float(thr_lgb_opt)
}
joblib.dump(optuna_thresholds, '../models/optuna_thresholds.pkl')

print('Optuna optimized models saved!')
print('  models/xgb_optuna.pkl')
print('  models/lgb_optuna.pkl')
print('  models/optuna_best_params.pkl')
print('  models/optuna_thresholds.pkl')
print(f'\nXGBoost Optuna best params:')
for k, v in xgb_study.best_params.items():
    print(f'  {k}: {v}')
print(f'\nLightGBM Optuna best params:')
for k, v in lgb_study.best_params.items():
    print(f'  {k}: {v}')
print(f'\nOptimal thresholds:')
for k, v in optuna_thresholds.items():
    print(f'  {k}: {v:.4f}')

Optuna optimized models saved!
  models/xgb_optuna.pkl
  models/lgb_optuna.pkl
  models/optuna_best_params.pkl
  models/optuna_thresholds.pkl

XGBoost Optuna best params:
  n_estimators: 499
  max_depth: 10
  learning_rate: 0.08292019680569757
  subsample: 0.6065850351483025
  colsample_bytree: 0.985981607194762
  min_child_weight: 1
  scale_pos_weight: 7

LightGBM Optuna best params:
  n_estimators: 438
  max_depth: 10
  learning_rate: 0.04620201864166716
  num_leaves: 83
  subsample: 0.8880426363593802
  colsample_bytree: 0.7050800343541103
  min_child_samples: 45
  scale_pos_weight: 5
  reg_alpha: 0.08162398418764394
  reg_lambda: 0.511674345354574

Optimal thresholds:
  xgb_optuna: 0.0335
  lgb_optuna: 0.2457


In [14]:
# Final Summary
print(f'\nModels saved:')
print(f'  models/xgb_optuna.pkl')
print(f'  models/lgb_optuna.pkl')
print(f'  models/optuna_best_params.pkl')
print(f'  models/optuna_thresholds.pkl')
print(f'  dashboard/optuna_comparison.csv')

print(f'\nOptuna Results Summary:')
print(f'\n{"Model":<20} {"CV_AUC":>10} {"Val_AUC":>10} {"Recall":>10}')
print('-' * 55)
print(f'{"XGBoost manual":<20} {"0.8823":>10} {"0.5953":>10} {"0.4408":>10}')
print(f'{"LightGBM manual":<20} {"0.8818":>10} {"0.6068":>10} {"0.4594":>10}')
print(f'{"XGBoost Optuna":<20} {xgb_study.best_value:>10.4f} {auc_xgb_opt:>10.4f} {rec_xgb_opt:>10.4f}')
print(f'{"LightGBM Optuna":<20} {lgb_study.best_value:>10.4f} {auc_lgb_opt:>10.4f} {rec_lgb_opt:>10.4f}')

print(f'\nKey Findings:')
print(f'  1. Optuna improved XGBoost CV from 0.8823 to {xgb_study.best_value:.4f}')
print(f'  2. Optuna improved LightGBM CV from 0.8818 to {lgb_study.best_value:.4f}')
print(f'  3. XGBoost Optuna best recall: {rec_xgb_opt:.4f} — catches most high risk patients')
print(f'  4. LightGBM Optuna best ROC-AUC: {auc_lgb_opt:.4f}')
print(f'  5. Bayesian optimization found better params than manual tuning')


Models saved:
  models/xgb_optuna.pkl
  models/lgb_optuna.pkl
  models/optuna_best_params.pkl
  models/optuna_thresholds.pkl
  dashboard/optuna_comparison.csv

Optuna Results Summary:

Model                    CV_AUC    Val_AUC     Recall
-------------------------------------------------------
XGBoost manual           0.8823     0.5953     0.4408
LightGBM manual          0.8818     0.6068     0.4594
XGBoost Optuna           0.8913     0.5893     0.6081
LightGBM Optuna          0.8839     0.6102     0.5534

Key Findings:
  1. Optuna improved XGBoost CV from 0.8823 to 0.8913
  2. Optuna improved LightGBM CV from 0.8818 to 0.8839
  3. XGBoost Optuna best recall: 0.6081 — catches most high risk patients
  4. LightGBM Optuna best ROC-AUC: 0.6102
  5. Bayesian optimization found better params than manual tuning
